# Save arrays
np.save(os.path.join(OUTPUT_DIR, "train_X.npy"), X_train_pdd)
np.save(os.path.join(OUTPUT_DIR, "train_y.npy"), y_train_pdd)
np.save(os.path.join(OUTPUT_DIR, "val_X.npy"), X_val_pdd)
np.save(os.path.join(OUTPUT_DIR, "val_y.npy"), y_val_pdd)
np.save(os.path.join(OUTPUT_DIR, "test_X.npy"), X_test_pdd)
np.save(os.path.join(OUTPUT_DIR, "test_y.npy"), y_test_pdd)

# Save class names
class_names_path = os.path.join(OUTPUT_DIR, "class_names.txt")
with open(class_names_path, "w", encoding="utf-8") as f:
    for name in class_names_pdd:
        f.write(name + "\n")

print(f"\nSaved preprocessed Plant Disease Detection data to '{OUTPUT_DIR}'")
print(f"Class names written to {class_names_path}")


# Cell 1 – Importlar ve ayarlar

In [1]:
import os
import random
from PIL import Image
import numpy as np

# -------------------
# Configuration
# -------------------

DATA_ROOT = "../Dataset/plantdoc_converted"

OUTPUT_DIR = "preprocessed_plantdoc"

IMG_SIZE = 32
RANDOM_SEED = 42

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
MAX_IMAGES_PER_CLASS = None

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# Cell 2 – Yardımcı fonksiyonlar (sınıf isimleri + görüntü vektörü)

In [2]:
def get_class_names(root_dir):
    class_names = []
    for item in os.listdir(root_dir):
        full_path = os.path.join(root_dir, item)
        if os.path.isdir(full_path):
            class_names.append(item)
    class_names = sorted(class_names)
    return class_names


def load_image_as_vector(path, img_size=IMG_SIZE):
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize((img_size, img_size))
        pixels = list(img.getdata())
        vector = [p / 255.0 for p in pixels]
        return vector


def build_splits_for_dataset(root_dir, train_ratio, val_ratio, max_per_class=None):
    class_names = get_class_names(root_dir)
    print("Found classes:")
    for idx, name in enumerate(class_names):
        print(f"{idx:2d} -> {name}")
    print()

    X_train, y_train = [], []
    X_val, y_val = [], []
    X_test, y_test = [], []

    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(root_dir, class_name)
        image_files = [
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        if max_per_class is not None:
            random.shuffle(image_files)
            image_files = image_files[:max_per_class]
        else:
            random.shuffle(image_files)

        n_total = len(image_files)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        n_test = n_total - n_train - n_val

        train_files = image_files[:n_train]
        val_files = image_files[n_train:n_train + n_val]
        test_files = image_files[n_train + n_val:]

        print(f"Class '{class_name}' (idx {class_idx}): "
              f"total={n_total}, train={len(train_files)}, "
              f"val={len(val_files)}, test={len(test_files)}")

        for fname in train_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_train.append(x_vec)
            y_train.append(class_idx)

        for fname in val_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_val.append(x_vec)
            y_val.append(class_idx)

        for fname in test_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_test.append(x_vec)
            y_test.append(class_idx)

    print("\nTotal samples:")
    print("Train:", len(X_train))
    print("Val  :", len(X_val))
    print("Test :", len(X_test))

    return (
        np.array(X_train, dtype=np.float32),
        np.array(y_train, dtype=np.int64),
        np.array(X_val, dtype=np.float32),
        np.array(y_val, dtype=np.int64),
        np.array(X_test, dtype=np.float32),
        np.array(y_test, dtype=np.int64),
        class_names,
    )



# Çalıştır ve kaydet


In [3]:
(
    X_train_pd,
    y_train_pd,
    X_val_pd,
    y_val_pd,
    X_test_pd,
    y_test_pd,
    class_names_pd,
) = build_splits_for_dataset(
    DATA_ROOT,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    max_per_class=MAX_IMAGES_PER_CLASS,
)

print("\nShapes:")
print("X_train:", X_train_pd.shape, "y_train:", y_train_pd.shape)
print("X_val  :", X_val_pd.shape, "y_val  :", y_val_pd.shape)
print("X_test :", X_test_pd.shape, "y_test :", y_test_pd.shape)

# Save arrays
np.save(os.path.join(OUTPUT_DIR, "train_X.npy"), X_train_pd)
np.save(os.path.join(OUTPUT_DIR, "train_y.npy"), y_train_pd)
np.save(os.path.join(OUTPUT_DIR, "val_X.npy"), X_val_pd)
np.save(os.path.join(OUTPUT_DIR, "val_y.npy"), y_val_pd)
np.save(os.path.join(OUTPUT_DIR, "test_X.npy"), X_test_pd)
np.save(os.path.join(OUTPUT_DIR, "test_y.npy"), y_test_pd)

# Save class names
class_names_path = os.path.join(OUTPUT_DIR, "class_names.txt")
with open(class_names_path, "w", encoding="utf-8") as f:
    for name in class_names_pd:
        f.write(name + "\n")

print(f"\nSaved preprocessed PlantDoc (converted) data to '{OUTPUT_DIR}'")
print(f"Class names written to {class_names_path}")



Found classes:
 0 -> Apple Scab Leaf
 1 -> Apple leaf
 2 -> Apple rust leaf
 3 -> Bell_pepper leaf
 4 -> Bell_pepper leaf spot
 5 -> Blueberry leaf
 6 -> Cherry leaf
 7 -> Corn Gray leaf spot
 8 -> Corn leaf blight
 9 -> Corn rust leaf
10 -> Peach leaf
11 -> Potato leaf
12 -> Potato leaf early blight
13 -> Potato leaf late blight
14 -> Raspberry leaf
15 -> Soyabean leaf
16 -> Soybean leaf
17 -> Squash Powdery mildew leaf
18 -> Strawberry leaf
19 -> Tomato Early blight leaf
20 -> Tomato Septoria leaf spot
21 -> Tomato leaf
22 -> Tomato leaf bacterial spot
23 -> Tomato leaf late blight
24 -> Tomato leaf mosaic virus
25 -> Tomato leaf yellow virus
26 -> Tomato mold leaf
27 -> Tomato two spotted spider mites leaf
28 -> grape leaf
29 -> grape leaf black rot

Class 'Apple Scab Leaf' (idx 0): total=93, train=65, val=13, test=15
Class 'Apple leaf' (idx 1): total=91, train=63, val=13, test=15
Class 'Apple rust leaf' (idx 2): total=88, train=61, val=13, test=14
Class 'Bell_pepper leaf' (idx 3): 